# Hierarchical Clustering for Grafana Logs

This notebook performs Hierarchical (Agglomerative) clustering on the extracted features from Grafana logs.

## Hierarchical Clustering Algorithm:
- **Type**: Hierarchical agglomerative clustering
- **Advantages**: No need to predefine k, produces dendrogram for visualization, captures hierarchy in data
- **Disadvantages**: Computationally expensive (O(n³) time, O(n²) space), sensitive to noise and outliers

## Linkage Methods:
- **Ward**: Minimizes within-cluster variance (good for compact clusters)
- **Average**: Average distance between all pairs
- **Complete**: Maximum distance between clusters (good for avoiding chain effect)
- **Single**: Minimum distance (prone to chaining)

## Steps:
1. Load feature matrices
2. Compute dendrograms for different linkage methods
3. Determine optimal number of clusters
4. Perform hierarchical clustering
5. Evaluate clustering quality
6. Visualize clusters and dendrogram
7. Analyze cluster characteristics
8. Benchmark performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
import time
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## 1. Load Feature Matrices

In [ ]:
# Load scaled features
X_scaled = pd.read_csv('features_scaled.csv')
print(f"Scaled features shape: {X_scaled.shape}")

# For hierarchical clustering, we'll use a sample if the dataset is too large
# Hierarchical clustering has O(n²) memory complexity
MAX_SAMPLES = 10000

if len(X_scaled) > MAX_SAMPLES:
    print(f"\nDataset is large ({len(X_scaled)} samples).")
    print(f"Sampling {MAX_SAMPLES} samples for hierarchical clustering...")
    sample_indices = np.random.choice(len(X_scaled), MAX_SAMPLES, replace=False)
    X_sample = X_scaled.iloc[sample_indices].copy()
    is_sampled = True
else:
    X_sample = X_scaled.copy()
    sample_indices = np.arange(len(X_scaled))
    is_sampled = False

print(f"\nUsing {len(X_sample)} samples for hierarchical clustering")

# Load PCA features
X_pca = pd.read_csv('features_pca.csv')
X_pca_sample = X_pca.iloc[sample_indices]

# Load metadata
metadata = pd.read_csv('metadata.csv')
metadata_sample = metadata.iloc[sample_indices].reset_index(drop=True)

# Convert to numpy arrays
X_array = X_sample.values
X_pca_array = X_pca_sample.values

print("\nData loaded successfully!")

## 2. Compute Linkage Matrix and Dendrogram

In [ ]:
# Test different linkage methods
linkage_methods = ['ward', 'average', 'complete']
linkage_matrices = {}

print("Computing linkage matrices...\n")

for method in linkage_methods:
    print(f"Computing {method} linkage...", end=' ')
    start_time = time.time()
    Z = linkage(X_array, method=method)
    elapsed = time.time() - start_time
    linkage_matrices[method] = Z
    print(f"Done in {elapsed:.2f}s")

print("\nLinkage computation complete!")

In [ ]:
# Plot dendrograms for different linkage methods
fig, axes = plt.subplots(len(linkage_methods), 1, figsize=(16, 6*len(linkage_methods)))

if len(linkage_methods) == 1:
    axes = [axes]

for idx, method in enumerate(linkage_methods):
    dendrogram(linkage_matrices[method], ax=axes[idx], 
               truncate_mode='lastp', p=50,  # Show only last 50 merges
               no_labels=True)
    axes[idx].set_title(f'Dendrogram - {method.capitalize()} Linkage', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Sample Index (or Cluster Size)', fontsize=12)
    axes[idx].set_ylabel('Distance', fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hierarchical_dendrograms.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Determine Optimal Number of Clusters

In [ ]:
# Test different numbers of clusters for each linkage method
K_range = range(2, 21)
results = {method: {'silhouette': [], 'davies_bouldin': [], 'calinski_harabasz': []} 
           for method in linkage_methods}

print("Testing different numbers of clusters...")
print("This may take several minutes...\n")

for method in linkage_methods:
    print(f"\nLinkage method: {method}")
    for k in K_range:
        print(f"  k={k}...", end=' ')
        
        # Get cluster labels from dendrogram
        labels = fcluster(linkage_matrices[method], k, criterion='maxclust')
        
        # Calculate metrics
        sil_score = silhouette_score(X_array, labels)
        db_score = davies_bouldin_score(X_array, labels)
        ch_score = calinski_harabasz_score(X_array, labels)
        
        results[method]['silhouette'].append(sil_score)
        results[method]['davies_bouldin'].append(db_score)
        results[method]['calinski_harabasz'].append(ch_score)
        
        print(f"Silhouette: {sil_score:.4f}")

print("\nOptimization complete!")

In [ ]:
# Plot evaluation metrics for different linkage methods
fig, axes = plt.subplots(len(linkage_methods), 3, figsize=(18, 6*len(linkage_methods)))

if len(linkage_methods) == 1:
    axes = axes.reshape(1, -1)

for idx, method in enumerate(linkage_methods):
    # Silhouette Score
    axes[idx, 0].plot(K_range, results[method]['silhouette'], 'go-', linewidth=2, markersize=8)
    axes[idx, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[idx, 0].set_ylabel('Silhouette Score', fontsize=12)
    axes[idx, 0].set_title(f'{method.capitalize()} - Silhouette Score', fontsize=13, fontweight='bold')
    axes[idx, 0].grid(True, alpha=0.3)
    best_k = K_range[np.argmax(results[method]['silhouette'])]
    axes[idx, 0].axvline(x=best_k, color='r', linestyle='--', label=f'Best k={best_k}')
    axes[idx, 0].legend()
    
    # Davies-Bouldin Index
    axes[idx, 1].plot(K_range, results[method]['davies_bouldin'], 'ro-', linewidth=2, markersize=8)
    axes[idx, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[idx, 1].set_ylabel('Davies-Bouldin Index', fontsize=12)
    axes[idx, 1].set_title(f'{method.capitalize()} - Davies-Bouldin (Lower Better)', fontsize=13, fontweight='bold')
    axes[idx, 1].grid(True, alpha=0.3)
    best_k = K_range[np.argmin(results[method]['davies_bouldin'])]
    axes[idx, 1].axvline(x=best_k, color='g', linestyle='--', label=f'Best k={best_k}')
    axes[idx, 1].legend()
    
    # Calinski-Harabasz Score
    axes[idx, 2].plot(K_range, results[method]['calinski_harabasz'], 'mo-', linewidth=2, markersize=8)
    axes[idx, 2].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[idx, 2].set_ylabel('Calinski-Harabasz Score', fontsize=12)
    axes[idx, 2].set_title(f'{method.capitalize()} - Calinski-Harabasz (Higher Better)', fontsize=13, fontweight='bold')
    axes[idx, 2].grid(True, alpha=0.3)
    best_k = K_range[np.argmax(results[method]['calinski_harabasz'])]
    axes[idx, 2].axvline(x=best_k, color='r', linestyle='--', label=f'Best k={best_k}')
    axes[idx, 2].legend()

plt.tight_layout()
plt.savefig('hierarchical_optimal_k.png', dpi=300, bbox_inches='tight')
plt.show()

# Print best k for each method
print("\nOptimal k suggestions by linkage method:")
print("=" * 80)
for method in linkage_methods:
    best_sil_k = K_range[np.argmax(results[method]['silhouette'])]
    best_db_k = K_range[np.argmin(results[method]['davies_bouldin'])]
    best_ch_k = K_range[np.argmax(results[method]['calinski_harabasz'])]
    
    print(f"\n{method.capitalize()} Linkage:")
    print(f"  - Best Silhouette: k={best_sil_k} (score: {max(results[method]['silhouette']):.4f})")
    print(f"  - Best Davies-Bouldin: k={best_db_k} (score: {min(results[method]['davies_bouldin']):.4f})")
    print(f"  - Best Calinski-Harabasz: k={best_ch_k} (score: {max(results[method]['calinski_harabasz']):.2f})")

## 4. Select Best Linkage Method and Optimal k

In [ ]:
# Select method with best average silhouette score
avg_silhouette = {method: np.mean(results[method]['silhouette']) for method in linkage_methods}
best_method = max(avg_silhouette, key=avg_silhouette.get)
optimal_k = K_range[np.argmax(results[best_method]['silhouette'])]

print(f"Selected linkage method: {best_method}")
print(f"Selected optimal k: {optimal_k}")
print(f"\nAverage silhouette scores by method:")
for method, score in avg_silhouette.items():
    print(f"  {method}: {score:.4f}")

## 5. Perform Final Hierarchical Clustering

In [ ]:
# Perform hierarchical clustering with optimal parameters
print(f"Performing Hierarchical clustering with {best_method} linkage and k={optimal_k}...")

start_time = time.time()
hierarchical = AgglomerativeClustering(n_clusters=optimal_k, linkage=best_method)
cluster_labels = hierarchical.fit_predict(X_array)
clustering_time = time.time() - start_time

print(f"Clustering completed in {clustering_time:.2f} seconds")
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} samples ({count/len(cluster_labels)*100:.2f}%)")

## 6. Evaluate Clustering Quality

In [ ]:
# Calculate evaluation metrics
silhouette = silhouette_score(X_array, cluster_labels)
davies_bouldin = davies_bouldin_score(X_array, cluster_labels)
calinski_harabasz = calinski_harabasz_score(X_array, cluster_labels)

print("=" * 80)
print("HIERARCHICAL CLUSTERING EVALUATION METRICS")
print("=" * 80)
print(f"\nLinkage method: {best_method}")
print(f"Number of clusters: {optimal_k}")
print(f"Number of samples: {len(cluster_labels):,}")
print(f"Clustering time: {clustering_time:.2f} seconds")
print(f"\nQuality Metrics:")
print(f"  - Silhouette Score: {silhouette:.4f} (range: [-1, 1], higher is better)")
print(f"  - Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
print(f"  - Calinski-Harabasz Score: {calinski_harabasz:.2f} (higher is better)")
print("\n" + "=" * 80)

# Store metrics for comparison
hierarchical_metrics = {
    'algorithm': 'Hierarchical',
    'linkage_method': best_method,
    'n_clusters': optimal_k,
    'silhouette_score': silhouette,
    'davies_bouldin_index': davies_bouldin,
    'calinski_harabasz_score': calinski_harabasz,
    'clustering_time': clustering_time,
    'n_samples': len(cluster_labels),
    'is_sampled': is_sampled
}

## 7. Visualize Clusters

In [ ]:
# Visualize clusters using PCA
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PC1 vs PC2
scatter1 = axes[0].scatter(X_pca_array[:, 0], X_pca_array[:, 1], 
                           c=cluster_labels, cmap='viridis', alpha=0.6, s=30)
axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title(f'Hierarchical Clusters ({best_method}) - PC1 vs PC2', fontsize=14, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: PC2 vs PC3
scatter2 = axes[1].scatter(X_pca_array[:, 1], X_pca_array[:, 2], 
                           c=cluster_labels, cmap='viridis', alpha=0.6, s=30)
axes[1].set_xlabel('Second Principal Component', fontsize=12)
axes[1].set_ylabel('Third Principal Component', fontsize=12)
axes[1].set_title(f'Hierarchical Clusters ({best_method}) - PC2 vs PC3', fontsize=14, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig('hierarchical_clusters_pca.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3D visualization
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(X_pca_array[:, 0], X_pca_array[:, 1], X_pca_array[:, 2],
                     c=cluster_labels, cmap='viridis', alpha=0.6, s=20)

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_zlabel('PC3', fontsize=12)
ax.set_title(f'Hierarchical Clusters ({best_method}) in 3D PCA Space', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Cluster', shrink=0.6)

plt.tight_layout()
plt.savefig('hierarchical_clusters_3d.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Cluster Characteristics Analysis

In [ ]:
# Add cluster labels to metadata
metadata_sample['cluster'] = cluster_labels

# Analyze cluster characteristics
print("Cluster Characteristics:\n")
print("=" * 80)

for cluster_id in range(optimal_k):
    cluster_data = metadata_sample[metadata_sample['cluster'] == cluster_id]
    
    print(f"\nCluster {cluster_id} (n={len(cluster_data)}):")
    print("-" * 80)
    
    # Top panels
    print("  Top 5 Panels:")
    top_panels = cluster_data['panel_title'].value_counts().head(5)
    for panel, count in top_panels.items():
        print(f"    - {panel}: {count} ({count/len(cluster_data)*100:.1f}%)")
    
    # Top services
    print("\n  Top 5 Services:")
    top_services = cluster_data['service'].value_counts().head(5)
    for service, count in top_services.items():
        print(f"    - {service}: {count} ({count/len(cluster_data)*100:.1f}%)")
    
    # Value statistics
    print("\n  Value Statistics:")
    print(f"    - Mean: {cluster_data['value'].mean():.2f}")
    print(f"    - Median: {cluster_data['value'].median():.2f}")
    print(f"    - Std: {cluster_data['value'].std():.2f}")
    print(f"    - Min: {cluster_data['value'].min():.2f}")
    print(f"    - Max: {cluster_data['value'].max():.2f}")

print("\n" + "=" * 80)

In [ ]:
# Visualize cluster characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Cluster size distribution
cluster_sizes = metadata_sample['cluster'].value_counts().sort_index()
axes[0, 0].bar(cluster_sizes.index, cluster_sizes.values, color='steelblue')
axes[0, 0].set_xlabel('Cluster ID', fontsize=12)
axes[0, 0].set_ylabel('Number of Samples', fontsize=12)
axes[0, 0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Value distribution by cluster
metadata_sample.boxplot(column='value', by='cluster', ax=axes[0, 1])
axes[0, 1].set_xlabel('Cluster ID', fontsize=12)
axes[0, 1].set_ylabel('Value', fontsize=12)
axes[0, 1].set_title('Value Distribution by Cluster', fontsize=14, fontweight='bold')
plt.sca(axes[0, 1])
plt.xticks(rotation=0)

# Panel distribution across clusters
top_panels = metadata_sample['panel_title'].value_counts().head(8).index
panel_cluster_counts = pd.crosstab(metadata_sample['panel_title'], metadata_sample['cluster'])
panel_cluster_counts.loc[top_panels].plot(kind='bar', stacked=True, ax=axes[1, 0], colormap='viridis')
axes[1, 0].set_xlabel('Panel Title', fontsize=12)
axes[1, 0].set_ylabel('Count', fontsize=12)
axes[1, 0].set_title('Top Panels Distribution Across Clusters', fontsize=14, fontweight='bold')
axes[1, 0].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.sca(axes[1, 0])
plt.xticks(rotation=45, ha='right')

# Service distribution across clusters
top_services = metadata_sample['service'].value_counts().head(8).index
service_cluster_counts = pd.crosstab(metadata_sample['service'], metadata_sample['cluster'])
service_cluster_counts.loc[top_services].plot(kind='bar', stacked=True, ax=axes[1, 1], colormap='viridis')
axes[1, 1].set_xlabel('Service', fontsize=12)
axes[1, 1].set_ylabel('Count', fontsize=12)
axes[1, 1].set_title('Top Services Distribution Across Clusters', fontsize=14, fontweight='bold')
axes[1, 1].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.sca(axes[1, 1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('hierarchical_cluster_characteristics.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Save Results

In [ ]:
# Save cluster assignments
results_df = metadata_sample.copy()
results_df.to_csv('hierarchical_cluster_assignments.csv', index=False)
print("Cluster assignments saved to: hierarchical_cluster_assignments.csv")

# Save metrics
import json
with open('hierarchical_metrics.json', 'w') as f:
    json.dump(hierarchical_metrics, f, indent=2)
print("Metrics saved to: hierarchical_metrics.json")

# Save model
import pickle
with open('hierarchical_model.pkl', 'wb') as f:
    pickle.dump(hierarchical, f)
print("Model saved to: hierarchical_model.pkl")

# Save linkage matrix
with open('hierarchical_linkage_matrix.pkl', 'wb') as f:
    pickle.dump(linkage_matrices[best_method], f)
print("Linkage matrix saved to: hierarchical_linkage_matrix.pkl")

print("\nHierarchical clustering complete!")